## BERT 전이학습

In [1]:
from tensorflow.keras.utils import get_file

ratings_train_path = get_file('ratings_train.txt', 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt')
ratings_test_path = get_file('ratings_test.txt', 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt')

14628807/14628807 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4893335/4893335 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
import pandas as pd

# 데이터(tsv) 로드 -> DataFrame으로 변환
ratings_train_df = pd.read_csv(ratings_train_path, sep = '\t')
ratings_test_df = pd.read_csv(ratings_test_path, sep = '\t')

display(ratings_train_df.head())
display(ratings_test_df.head())

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [3]:
ratings_train_df = ratings_train_df.dropna(how='any')
ratings_test_df = ratings_test_df.dropna(how='any')

print(ratings_train_df.isnull().sum(), ratings_test_df.isnull().sum())

id          0
document    0
label       0
dtype: int64 id          0
document    0
label       0
dtype: int64


In [4]:
ratings_train_df = ratings_train_df.sample(n=15_000, random_state= 0)
ratings_test_df = ratings_test_df.sample(n=5_000, random_state= 0)

ratings_train_df['label'].value_counts(), ratings_test_df['label'].value_counts()

(label
 0    7512
 1    7488
 Name: count, dtype: int64,
 label
 0    2532
 1    2468
 Name: count, dtype: int64)

In [5]:
# 텍스트/라벨을 리스트로 변환 (학습/테스트 데이터)
X_train = ratings_train_df['document'].values.tolist()
y_train = ratings_train_df['label'].values.tolist()

X_test = ratings_test_df['document'].values.tolist()
y_test = ratings_test_df['label'].values.tolist()

### 토크나이저 /모델 준비
- bert 한국어 버전 사전학습 보델 klue/bert-base

In [6]:
from transformers import AutoModel, AutoTokenizer
from transformers import BertForSequenceClassification      # 클래스 분류용 BERT 모델

model_name = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)   # 토크나이저 (토큰화 규칙/어휘 사전)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
X_train = tokenizer(X_train, padding= True, truncation = True, return_tensors='pt')
X_test = tokenizer(X_test, padding= True, truncation= True, return_tensors='pt')

X_train[:3], X_test[:3]

([Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])],
 [Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])])

In [8]:
print(X_train['input_ids'][0])      # 토큰이 정수 id로 변환된 시퀀스
print(X_train['attention_mask'][0]) # 실제 토큰 = 1 , 패딩 = 0 으로 구분한 마스크
print(X_train['token_type_ids'][0]) # 문장 구분 ID

tensor([    2,  1800,  2178,   860,  3629, 16516,  2031,    18,    18,    18,
        14242,  2205,  2062,     3,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

## 데이터 파이프라인 생성

In [9]:
import torch 
from torch.utils.data import Dataset, DataLoader

# BERT 입력 데이터와 라벨을 함께 관리하는 Dataset 클래스
class NSMCDataset(Dataset):
    def __init__(self,encodings,labels):
        self.encodings = encodings  # 토큰화 결과(input_ids, attention_mask등) 저장
        self.labels = labels        # 정답 라벨

    def __len__(self):
        return len(self.labels)

    # idx 번째 샘플의 토큰화 결과(input_ids, attention_mask등)을 딕셔너리 형태로 반환
    def __getitem__(self, idx):
        item = { key:value[idx] for key,value in self.encodings.items()}

        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

In [10]:
# Pytorch Dataset 생성

train_dataset = NSMCDataset(X_train,y_train)
test_dataset = NSMCDataset(X_test,y_test)

# 학습 데이터 : 셔플 후 64개씩 배치 / 테스트 데이터 : 셔플없이 64개씩 배치
train_dataloader = DataLoader(train_dataset, batch_size= 64,shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size= 64,shuffle=False)

In [11]:
# 파인튜닝 설정
import torch
from transformers import get_scheduler  # 학습률 스케줄러 생성 함수

epochs = 5

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay= 0.1)

num_train_steps = len(train_dataloader) * epochs   # 전체 학습 step수

# warmup step 수 : 전체 학습 step에서 10%는 학습률을 점진적으로 증가
# - 사전학습 fine-tuning시에는 초반에 LR가 크면 학습이 불안정해지는 경우가 많아서 0에서부터 lr까지 점차적으로 증가
num_warmup_steps = int(num_train_steps * 0.1)   

lr_scheduler = get_scheduler(
    name = 'linear',            # Warmup이후 학습률 선형적으로 감소
    optimizer = optimizer,
    num_warmup_steps= num_warmup_steps,
    num_training_steps= num_train_steps
)

In [12]:
from tqdm.auto import tqdm

# 학습
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in tqdm(range(epochs)):
    model.train()

    total_loss = 0

    for batch in tqdm(train_dataloader, desc =f"{epoch+1}/{epoch}", leave = False ):
        batch = {k:v.to(device) for k,v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        lr_scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch : {epoch+1}/ {epochs} : Loss : {avg_loss:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

1/0:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch : 1/ 5 : Loss : 0.4032


2/1:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch : 2/ 5 : Loss : 0.2154


...

Epoch : 3/ 5 : Loss : 0.0997


4/3:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch : 4/ 5 : Loss : 0.0445


5/4:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch : 5/ 5 : Loss : 0.0239


In [13]:
model.save_pretrained('nsmc_model/bert-base')       # 학습된 모델 저장
tokenizer.save_pretrained('nsmc_model/bert-base')   # 모델에서 사용한 토크나이저 저장

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



('nsmc_model/bert-base/tokenizer_config.json',
 'nsmc_model/bert-base/tokenizer.json')

In [14]:
model.config.id2label = {
    0 : '부정',
    1 : '긍정'
}

In [15]:
# 감성분석 파이프라인 생성
from transformers import TextClassificationPipeline

# 입력 텍스트 -> 토큰화 -> 모델 추론 -> 라벨/점수 반환 파이프라인
sentiment_classifier = TextClassificationPipeline(
    tokenizer = tokenizer,
    model = model,
    framwork = 'pt',        # Pytorch 기반 모델
    top_k = None            # 모든 클래스 라벨과 확률 반환
)

In [16]:
sentiment_classifier('이것은 제 생애 가장 훌룡한 영화입니다!! 대단합니다!')

[[{'label': '긍정', 'score': 0.9924307465553284},
  {'label': '부정', 'score': 0.007569223642349243}]]

## HuggingFace 업로드

In [20]:
from getpass import getpass     # 입력값을 화면에 표시하지않고 바로 받는 함수
from huggingface_hub import login   # Hugginface 로그인

HF_TOKEN = getpass("HF_TOKEN : ")   # AccessToken 입력
login(token = HF_TOKEN)             # 입력한 토큰으로 로그인

In [26]:
REPO_NAME = 'bert-base-nsmc'        # Hub에 업로드할 레포지토리 이름(내 계정/REPO_NAME 으로 생성)

# 학습된 모델과 토크나이저를 Hub Repo업로드 (임시폴더 사용, 토큰으로 인증)
model.push_to_hub(REPO_NAME, token = HF_TOKEN)
tokenizer.push_to_hub(REPO_NAME, token = HF_TOKEN)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...jpp0vt0/model.safetensors:   4%|3         | 16.0MB /  442MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/castleUk/bert-base-nsmc/commit/78b901623bc8ab0a7f0b91863d94687efa176402', commit_message='Upload tokenizer', commit_description='', oid='78b901623bc8ab0a7f0b91863d94687efa176402', pr_url=None, repo_url=RepoUrl('https://huggingface.co/castleUk/bert-base-nsmc', endpoint='https://huggingface.co', repo_type='model', repo_id='castleUk/bert-base-nsmc'), pr_revision=None, pr_num=None)

In [23]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification

HUB_NAME = 'castleUK/bert-base-nsmc'

tokenizer = AutoTokenizer.from_pretrained(HUB_NAME)
model = AutoModelForSequenceClassification.from_pretrained(HUB_NAME)

config.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/404 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [24]:
from transformers import pipeline

sentiment_classifier = pipeline('text-classification',model = HUB_NAME)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [25]:
sentiment_classifier([
    '한국 영화는 이래서 안돼~',
    '역시 봉감독이 최고야!',
    '진짜 정말로 강하게 재미없다.',
    ''
])

[{'label': '부정', 'score': 0.99989914894104},
 {'label': '긍정', 'score': 0.9765251874923706},
 {'label': '부정', 'score': 0.9998966455459595},
 {'label': '부정', 'score': 0.6994063854217529}]